# Germinal Antibody Design

![Germinal Antibody Design](https://proto-bio.github.io/proto-assets/images/tool/germinal/hero.png)

De novo epitope-targeted antibody design (VHH or scFv) using the Germinal pipeline ([Mille-Fragoso et al. 2025](https://www.biorxiv.org/content/10.1101/2025.09.19.677421)). This wrapper installs the upstream [`SantiagoMille/germinal`](https://github.com/SantiagoMille/germinal) repo at a pinned commit and runs one end-to-end campaign per call.

The pipeline is: **ColabDesign + AF2-Multimer hallucination → AbMPNN sequence redesign → Chai-1 / AF3 / Protenix structure validation → multi-stage filter cascade**.

> **Note: long-running tool.** This example uses a minimal smoke-test budget so it runs quickly and may produce zero accepted designs. A full design campaign that yields accepted binders runs many more trajectories and can take hours on GPU.

In [1]:
from pathlib import Path

from proto_tools.tools.binder_design import (
    GerminalConfig,
    GerminalInput,
    run_germinal_design,
)

## Input API — `GerminalInput`

| Field | Type | Default | Description |
|---|---|---|---|
| `target_pdb` | `str` | *required* | PDB file path or PDB-format string of the target. |
| `target_chain` | `str` | `"A"` | Target chain ID(s); comma-separated for multi-chain (e.g. `"A,B"`). |
| `binder_chain` | `str` | `"B"` | Designed binder chain ID. Matches Germinal's source default. |
| `hotspots` | `list[str]` | `[]` | Target hotspot residues, e.g. `["A37", "A39"]` (chain letter + 1-indexed resnum). |
| `target_name` | `str | None` | `None` | Short identifier; defaults to a hash of the PDB content. |
| `hotspot_residue` | `str | None` | `None` | Optional Chai-1 contact-restraint anchor residue (e.g. `"W40"`). |

## Configuration API — `GerminalConfig`

**Main parameters** (defaults match Germinal source `configs/config.yaml`):

| Field | Type | Default | Description |
|---|---|---|---|
| `design_type` | `Literal["vhh", "scfv"]` | `"vhh"` | Run preset selector. |
| `max_trajectories` | `int` | `10000` | Total trajectories to attempt. |
| `max_passing_designs` | `int` | `100` | Stop early once this many designs pass all final filters. |
| `structure_model` | `Literal["chai", "af3", "protenix"]` | `"chai"` | Cofolding backend. *Source defaults: `protenix` (VHH) / `af3` (scFv); we default to `chai` because it auto-installs.* |
| `plddt_threshold` | `float | None` | `None` | Override final `external_plddt` (preset wins if `None`). |
| `iptm_threshold` | `float | None` | `None` | Override final `external_iptm`. |
| `ipae_threshold` | `float | None` | `None` | Override final `external_pae` (Å). |
| `ptm_threshold` | `float | None` | `None` | Override final `external_ptm`. |
| `pdockq2_threshold` | `float | None` | `None` | Override final `pdockq2`. |

**Advanced parameters**:

| Field | Type | Default | Description |
|---|---|---|---|
| `max_hallucinated_trajectories` | `int` | `1000` | Stop after this many trajectories complete hallucination. |
| `germinal_overrides` | `dict[str, Any]` | `{}` | Free-form Hydra overrides for `run_germinal.py`, applied verbatim. |
| `filter_overrides` | `dict[str, dict[str, dict[str, Any]]]` | `{}` | Override filter YAMLs: `{"initial"|"final": {key: {"value": v, "operator": op}}}`. |

Inherited from `BaseConfig`: `device`, `timeout`, `seed`, `verbose`.

## Output API

**`GerminalOutput`** wraps a list of designs plus pipeline statistics:

| Field | Type | Description |
|---|---|---|
| `designs` | `list[GerminalDesign]` | All produced designs across stages. |
| `pipeline_stats` | `dict[str, int]` | Per-stage counts from Germinal's `failure_counts.csv`. |
| `num_accepted` | `int` (computed) | Designs that passed all final filters. |
| `num_designs` | `int` (computed) | Total designs returned. |

**`GerminalDesign`** — one antibody design:

| Field | Type | Description |
|---|---|---|
| `sequence_heavy` | `str` | Heavy / VHH chain sequence. |
| `sequence_light` | `str | None` | Light chain (scFv only). |
| `structure` | `Structure` | Predicted binder + target complex. |
| `metrics` | `GerminalDesignMetrics` | Per-design quality metrics. |
| `stage_passed` | `Literal["accepted", "redesign_candidate", "trajectory"]` | Highest pipeline stage reached. |
| `design_id` | `str` | Germinal's internal identifier. |
| `trajectory_index` / `mpnn_index` | `int` / `int` | Provenance: parent trajectory + AbMPNN sample index. |

**`GerminalDesignMetrics`** — `metric_spec` keys (subset; see source for full type/range info):

- *Trajectory metrics* (always populated): `plddt`, `ptm`, `i_ptm`, `i_pae`, `pae`, `loss`, `lm_ll`, `helix`, `beta_strand`
- *Filter metrics* (after filter stage): `clashes`, `sc_rmsd`, `binder_near_hotspot`, `cdr3_hotspot_contacts`, `percent_interface_cdr`, `interface_shape_comp`, `interface_hbonds`, `surface_hydrophobicity`, `interface_hydrophobicity`, `pdockq2`
- *External-validation metrics* (after structure validation): `external_plddt`, `external_iptm`, `external_ptm`, `external_pae`

Primary metric: `i_ptm`.

In [ ]:
# Resolve the in-tree PD-L1 test target by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pdb_path = _repo_root / "tests" / "dummy_data" / "pdl1.pdb"
print(f"Using target PDB: {pdb_path}")

inputs = GerminalInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    binder_chain="B",
    hotspots=["A56", "A66", "A115"],
    target_name="pdl1",
)

config = GerminalConfig(
    design_type="vhh",
    max_trajectories=2,
    max_passing_designs=1,
    structure_model="chai",
    germinal_overrides={
        "logits_steps": 20,
        "softmax_steps": 4,
        "search_steps": 1,
        "num_seqs": 2,
    },
)

result = run_germinal_design(inputs, config)
print(f"success={result.success}")
print(f"num_designs={result.num_designs}, num_accepted={result.num_accepted}")
print(f"pipeline_stats={result.pipeline_stats}")

In [ ]:
for design in result.designs[:3]:
    print(f"--- {design.design_id} ({design.stage_passed}) ---")
    print(f"sequence_heavy: {design.sequence_heavy}")
    if design.sequence_light:
        print(f"sequence_light: {design.sequence_light}")
    if 'i_ptm' in design.metrics:
        print(f"  i_ptm={design.metrics.i_ptm:.3f}")
    if 'plddt' in design.metrics:
        print(f"  plddt={design.metrics.plddt:.3f}")

# Export every produced design's PDB (one file per design_id under <export_path>/<name.lower()>/)
if result.designs:
    result.export(name="pdl1_designs", export_path="./outputs", file_format="pdb")
    print(f"Exported {result.num_designs} designs ({result.num_accepted} accepted) to ./outputs/pdl1_designs/")

In [2]:
# Advanced configuration: scFv mode + custom filter override + verbatim Hydra settings.
# This cell only constructs the config (no GPU / no subprocess) so the notebook
# can be executed end-to-end on hosts without GPU.
advanced_config = GerminalConfig(
    design_type="scfv",
    max_trajectories=10,
    max_passing_designs=2,
    structure_model="chai",
    iptm_threshold=0.80,  # Tighten the iptm filter beyond Germinal's preset (0.74)
    filter_overrides={
        "final": {
            # Require at least 4 H-bonds at the interface
            "interface_hbonds": {"value": 4, "operator": ">="},
        },
    },
    germinal_overrides={
        # Emphasise interface pTM in the hallucination loss
        "weights_iptm": 1.0,
        # Disable the helix bias for floppy / loopy CDR designs
        "weights_helix": 0.0,
    },
)

print("Advanced scFv config:")
print(f"  design_type        = {advanced_config.design_type}")
print(f"  structure_model    = {advanced_config.structure_model}")
print(f"  iptm_threshold     = {advanced_config.iptm_threshold}")
print(f"  filter_overrides   = {advanced_config.filter_overrides}")
print(f"  germinal_overrides = {advanced_config.germinal_overrides}")
# To actually run: result = run_germinal_design(inputs, advanced_config)

Advanced scFv config:
  design_type        = scfv
  structure_model    = chai
  iptm_threshold     = 0.8
  filter_overrides   = {'final': {'interface_hbonds': {'value': 4, 'operator': '>='}}}
  germinal_overrides = {'weights_iptm': 1.0, 'weights_helix': 0.0}
